# nova_harness 快速开始

本 Notebook 演示如何用 `nova_harness` 创建一次 Agent 会话并发起对话。

`nova_harness` 是 `nova_ai` + `nova_agent` 的高阶封装，提供：
- 自动会话持久化
- 设置管理
- 模型注册表
- 工具注册表
- 上下文压缩（可选）

运行前请设置 `VOLCENGINE_API_KEY`（或其他厂商 Key）。


In [ ]:
import os
import tempfile
from pathlib import Path

# 请替换为你的真实 API Key
os.environ["VOLCENGINE_API_KEY"] = "your-volcengine-api-key"

# 使用临时目录作为 agent 配置根目录，避免污染本地 ~/.nova
_tmp = tempfile.TemporaryDirectory()
os.environ["NOVA_AGENT_DIR"] = str(Path(_tmp.name) / "agent")


## 创建 AgentSession

`create_agent_session()` 会自动初始化配置目录、设置管理器、会话管理器和模型注册表。


In [ ]:
from nova_harness import create_agent_session

runtime = await create_agent_session()
session = runtime.session
print("AgentSession 已创建:", session)
print("当前模型:", session.agent.state.model.id if session.agent.state.model else None)


## 发起对话

调用 `session.prompt()` 发送用户消息，内部会自动驱动 `nova_agent.Agent` 完成多轮对话。


In [ ]:
# 订阅事件并打印关键节点
def on_event(event):
    if event.type in ("message_start", "message_end", "tool_execution_start", "tool_execution_end", "agent_end"):
        print(f"[{event.type}]")

unsubscribe = session.agent.subscribe(on_event)

await session.prompt("你好，请用一句话介绍自己。")
await session.agent.wait_for_idle()

unsubscribe()

# 查看最终消息
for msg in session.agent.state.messages:
    if msg.role == "assistant" and msg.content:
        print("\n最终回复:", msg.content[0].text)


## 继续对话

可以继续发送 `prompt`，也可以使用 `steer()` / `follow_up()` 管理队列。
